# Calculating kinetics values using Bayesian statistics

This notebook calculates $k_{on}$, $k_{1}$, $k_{2}$, and $k_{-1}$ using Bayesian statistics, as described in Ensign and Pande (2009).



In [1]:
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd
import glob
import os
import sklearn.cluster as clust
import sklearn.mixture as mix
import statistics
import scipy.stats
import math
import time
import sys

## Transition DataFrames
This portion of the notebook will produce dataframes of the key transition times for calculating kinetics values.

In [2]:
### CHANGE VARIABLES IN THIS BLOCK TO MATCH YOUR PROJECT

min_frames = 100 # 100 means 1 ns
fb_dist = 10.5
ec_dist = 21
SH3_conc = 3.80/1000 # in Molar
#box_vol = 535497.5431 # no salt
box_vol = 399687.2 # salt

# reads in the status csv this help to break down the plot that will be created into encounter, bound and unbound
stat = pd.read_csv('/data/jcardoso/binding_figures_800ArkA12/ArkA12/pair_wise_distances/binding/salt/status_files/ArkA12_binding_salt_status.csv')
#stat = pd.read_csv('/data/jcardoso/binding_figures_600ArkA12+800ArkA17+bound/ArkA12/pair_wise_distances/binding/no_salt/status_files/ArkA12_binding_no_salt_status.csv')

# this reads in the csv created to run the pair-wise analysis
binding_surface = pd.read_csv('/data/jcardoso/binding_figures_800ArkA12/ArkA12/pair_wise_distances/binding/salt/csv/combined/salt_binding_distances.csv')
#binding_surface = pd.read_csv('/data/jcardoso/binding_figures_600ArkA12+800ArkA17+bound/ArkA12/pair_wise_distances/binding/no_salt/csv/combined/no_salt_binding_distances.csv')

## Divide up data into independent simulations

num_sims = 50
sim_length = int(len(stat.index)/num_sims)

In [3]:
stat_bysim = pd.DataFrame(columns=['sim0']) # each sim in its own column, binding status for each frame in rows
bindsurf_bysim = pd.DataFrame(columns=['sim0']) # each sim in its own column, binding distance value for each frame in rows

for sim in range(num_sims):
    stat_bysim['sim'+str(sim)] = stat['status'][sim_length*sim:sim_length*(sim+1)].reset_index()['status']
    bindsurf_bysim['sim'+str(sim)] = binding_surface['avg'][sim_length*sim:sim_length*(sim+1)].reset_index()['avg']

## Making a data frame where the rows are the independent simulations and the columns are binding times

num_bound = 0
num_EC = 0

# Initialize transitions with NaNs in 'fully_bound' and 'encounter (ns)' columns
transitions = pd.DataFrame({'fully_bound (ns)': [float('nan')] * 0, 'encounter (ns)': [float('nan')] * 0})

# Initialize unbind_transitions with NaNs in 'unbound' column
unbind_transitions = pd.DataFrame({'unbound (ns)': [float('nan')] * 0})

## In continuation, in order to approximate kon we willl be adding the bound timeframes into a dataframe

for sim in stat_bysim.columns:
    # Find kon
    
    bound_indices = bindsurf_bysim.index[bindsurf_bysim[sim] < fb_dist].tolist()
    
    if len(bound_indices) < min_frames:
        new_row = pd.DataFrame([{'fully_bound (ns)': float('nan')}])
        transitions = pd.concat([transitions, new_row], ignore_index=True)      
    else:        
        first_bound = 0
        snapshot = 0        
        while first_bound == 0:
            min_met = False
            num_consec = 1
            consec = True
            while (num_consec < min_frames) & consec:
                if bound_indices[snapshot+1] > bound_indices[snapshot]+1:
                    consec = False
                else:
                    num_consec += 1
                    snapshot += 1         
            if num_consec >= min_frames:
                min_met = True
            if min_met == True:
                first_bound = bound_indices[snapshot]
            elif snapshot > len(bound_indices)-min_frames-1:
                first_bound = -1

            snapshot += 1
            
        if first_bound < 0:
            new_row = pd.DataFrame([{'fully_bound (ns)': float('nan')}])
            transitions = pd.concat([transitions, new_row], ignore_index=True)    
        else:
            num_bound += 1
            new_row = pd.DataFrame([{'fully_bound (ns)': first_bound * 0.01}])
             # converting to ns from frames
            transitions = pd.concat([transitions, new_row], ignore_index=True)

## Making a list of the timestamps that the simulations are considered unbound

# Initialize as empty DataFrames instead of numpy arrays
encounter_bind_times = pd.DataFrame(columns=['bind_time (ns)'])
unbind_times = pd.DataFrame(columns=['unbind_time (ns)'])

# Loop through each simulation column in stat_bysim
for sim in stat_bysim.columns:
    bound_indices = bindsurf_bysim.index[bindsurf_bysim[sim] < ec_dist].tolist()
    unbound_indices_all = bindsurf_bysim.index[bindsurf_bysim[sim] > ec_dist + 2]
    
    # Check if the length of bound indices is less than min_frames
    if len(bound_indices) < min_frames:
        # Create a DataFrame with NaN to append to encounter_bind_times
        new_row = pd.DataFrame([{'bind_time (ns)': float('nan')}])
        
        # Concatenate the NaN row to encounter_bind_times
        encounter_bind_times = pd.concat([encounter_bind_times, new_row], ignore_index=True)
    else:
        first_bound = 0
        snapshot = 0
        
        while first_bound == 0:

            min_met = False
            num_consec = 1
            consec = True

            while (num_consec < min_frames) and consec:
                if snapshot + 1 >= len(bound_indices):
                    consec = False  # End the loop if there's no next index to check
                elif bound_indices[snapshot + 1] > bound_indices[snapshot] + 1:
                    consec = False  # Break if the sequence is not consecutive
                else:
                    num_consec += 1  # Increase consecutive count
                    snapshot += 1    # Move to the next snapshot

            if num_consec >= min_frames:
                min_met = True

            if min_met == True:
                first_bound = bound_indices[snapshot]
            elif snapshot > len(bound_indices)-min_frames-1:
                first_bound = -1

            snapshot += 1
        
        if first_bound < 0:
            new_row = pd.DataFrame([{'bind_time (ns)': float('nan')}])
            encounter_bind_times = pd.concat([encounter_bind_times, new_row], ignore_index=True)
        else:
            num_EC += 1
            new_row = pd.DataFrame([{'bind_time (ns)': first_bound * 0.01}]) # converting to ns from frames
            encounter_bind_times = pd.concat([encounter_bind_times, new_row], ignore_index=True)
    
        unbound_indices =  unbound_indices_all[unbound_indices_all >= first_bound].tolist()
        
        if len(unbound_indices) < min_frames:
            pass
        else:
            first_unbound = 0
            snapshot = 0
        
            while first_unbound == 0:
            
                min_met = False
                num_consec = 1
                consec = True
            
                while (num_consec < min_frames) & consec:
                    if unbound_indices[snapshot+1] > unbound_indices[snapshot]+1:
                        consec = False
                    else:
                        num_consec += 1
                        snapshot += 1
                    
                if num_consec >= min_frames:
                    min_met = True

                if min_met == True:
                    first_unbound = unbound_indices[snapshot]
                elif snapshot > len(unbound_indices)-min_frames-1:
                    first_unbound = -1

                snapshot += 1
                   
            if first_unbound < 0:
                pass
            else:
                new_row = pd.DataFrame([{'unbind_time (ns)':  (first_unbound - first_bound) * 0.01}]) # converting to ns from frames
                unbind_times = pd.concat([unbind_times, new_row], ignore_index=True)

/tmp/ipykernel_581009/2852834070.py:113: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  encounter_bind_times = pd.concat([encounter_bind_times, new_row], ignore_index=True)
/tmp/ipykernel_581009/2852834070.py:150: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  unbind_times = pd.concat([unbind_times, new_row], ignore_index=True)


In [4]:
# Assign the encounter column
transitions['encounter (ns)'] = encounter_bind_times['bind_time (ns)']

# Now assign the 'unbind' column
unbind_transitions['unbound (ns)'] = unbind_times['unbind_time (ns)']

'''
## Save transitions data to csv file
transitions.to_csv('csv/transitions_data_ns.csv')
unbind_transitions.to_csv('csv/unbind_transitions_data_ns.csv')

stat_bysim.to_csv('csv/status_by_sim.csv')
'''

"\n## Save transitions data to csv file\ntransitions.to_csv('csv/transitions_data_ns.csv')\nunbind_transitions.to_csv('csv/unbind_transitions_data_ns.csv')\n\nstat_bysim.to_csv('csv/status_by_sim.csv')\n"

In [5]:
transitions

,fully_bound (ns),encounter (ns)
0,487.33,47.92
1,NaN,710.78
2,NaN,272.33
3,NaN,125.16
4,NaN,93.80
5,NaN,249.61
6,NaN,293.42
7,NaN,165.66
8,NaN,377.78
9,NaN,727.13


In [6]:
#new_df = df.loc[df['Column_B'].isna(), ['Column_A']].copy()
ec_transitions_never_bound = transitions.loc[transitions['fully_bound (ns)'].isna(), ['encounter (ns)']].copy().reset_index()

In [7]:
len(ec_transitions_never_bound)

41

## Kinetics Summary

This portion of the notebook calculates kinetics values from the transition dataframes.

In [8]:
transitions_data = transitions
'''
transitions_data = pd.read_csv('csv/transitions_data_ns.csv')
stat_bysim = pd.read_csv('csv/status_by_sim.csv')

num_bound = transitions_data['fully_bound (ns)'].count()
num_EC = transitions_data['encounter (ns)'].count()
'''

"\ntransitions_data = pd.read_csv('csv/transitions_data_ns.csv')\nstat_bysim = pd.read_csv('csv/status_by_sim.csv')\n\nnum_bound = transitions_data['fully_bound (ns)'].count()\nnum_EC = transitions_data['encounter (ns)'].count()\n"

## Bayesian Statistics (Jeffreys Prior)

In [9]:
## Bayesian statistics to find the kon value

sim_length_ns = sim_length * 0.01

tfi = transitions_data['fully_bound (ns)'].sum() #same as the total time before the complex is considered bound in the simulations that bind
ti = (num_sims - num_bound) * sim_length_ns #same as the total time of simulations that never bind

relv_traj_t = tfi + ti

bayesian_k = (num_bound)/relv_traj_t # in ns^-1
var_bayesian_k = (num_bound)/(relv_traj_t**2) # in ns^-2
std_bayesian_k = np.sqrt(var_bayesian_k) # in ns^-1

bayesian_kon = (bayesian_k/SH3_conc)*(10**9)  # unit conversion to M^-1 * s^-1
std_bayesian_kon = (std_bayesian_k/SH3_conc)*(10**9) # unit conversion to s^-1 * M^-1
se_bayesian_kon = std_bayesian_kon/np.sqrt(num_sims)
### k_on from MD sims (Bayesian)
kon_bayesian = bayesian_kon
kon_bayesian_err = std_bayesian_kon
print("bayesian_kon = " + format(kon_bayesian, '.2E') + ' ± ' + format(kon_bayesian_err, '.2E') +' s^-1 M^-1')

bayesian_kon = 5.27E+07 ± 1.76E+07 s^-1 M^-1


In [10]:
relv_traj_t

44909.91

In [11]:
## Bayesian statistics to find the k1 value

relv_traj_t_ec = transitions_data['encounter (ns)'].sum() #same as the total time before the complex is considered encounter in the simulations that bind

bayesian_k1 = (num_EC)/relv_traj_t_ec # in ns^-1
var_bayesian_k1 = (num_EC)/(relv_traj_t_ec**2) # in ns^-2
std_bayesian_k1 = np.sqrt(var_bayesian_k1) # in ns^-1

bayesian_k1 = (bayesian_k1/SH3_conc)*(10**9)  # unit conversion to M^-1 * s^-1
std_bayesian_k1 = (std_bayesian_k1/SH3_conc)*(10**9) # unit conversion to s^-1 * M^-1
se_bayesian_k1 = std_bayesian_k1/np.sqrt(num_sims)
### k1 from MD sims (Bayesian)
k1_bayesian = bayesian_k1
k1_bayesian_err = std_bayesian_k1
print("bayesian_k1 = " + format(k1_bayesian, '.2E') + ' ± ' + format(k1_bayesian_err, '.2E') +' s^-1 M^-1')

bayesian_k1 = 1.41E+09 ± 1.99E+08 s^-1 M^-1


In [12]:
relv_traj_t_ec

9344.069999999998

## Bayesian Statistics (Uniform Prior)

In [13]:
bayesian_k_U = (num_bound + 1)/relv_traj_t # in ns^-1
var_bayesian_k_U = (num_bound + 1)/(relv_traj_t**2) # in ns^-2
std_bayesian_k_U = np.sqrt(var_bayesian_k_U) # in ns^-1

bayesian_kon_U = (bayesian_k_U/SH3_conc)*(10**9)  # unit conversion to M^-1 * s^-1
std_bayesian_kon_U = (std_bayesian_k_U/SH3_conc)*(10**9) # unit conversion to s^-1 * M^-1
se_bayesian_kon_U = std_bayesian_kon_U/np.sqrt(num_sims)
### k_on from MD sims (Bayesian)
kon_bayesian_U = bayesian_kon_U
kon_bayesian_U_err = std_bayesian_kon_U
print("bayesian_kon (Uniform prior) = " + format(kon_bayesian_U, '.2E') + ' ± ' + format(kon_bayesian_U_err, '.2E') +' s^-1 M^-1')

bayesian_kon (Uniform prior) = 5.86E+07 ± 1.85E+07 s^-1 M^-1


In [14]:
## Bayesian statistics to find the k1 value

relv_traj_t_ec = transitions_data['encounter (ns)'].sum() #same as the total time before the complex is considered encounter in the simulations that bind

bayesian_k1_U = (num_EC + 1)/relv_traj_t_ec # in ns^-1
var_bayesian_k1_U = (num_EC + 1)/(relv_traj_t_ec**2) # in ns^-2
std_bayesian_k1_U = np.sqrt(var_bayesian_k1_U) # in ns^-1

bayesian_k1_U = (bayesian_k1_U/SH3_conc)*(10**9)  # unit conversion to M^-1 * s^-1
std_bayesian_k1_U = (std_bayesian_k1_U/SH3_conc)*(10**9) # unit conversion to s^-1 * M^-1
se_bayesian_k1_U = std_bayesian_k1_U/np.sqrt(num_sims)
### k1 from MD sims (Bayesian)
k1_bayesian_U = bayesian_k1_U
k1_bayesian_U_err = std_bayesian_k1_U
print("bayesian_k1 (Uniform prior) = " + format(k1_bayesian_U, '.2E') + ' ± ' + format(k1_bayesian_U_err, '.2E') +' s^-1 M^-1')

bayesian_k1 (Uniform prior) = 1.44E+09 ± 2.01E+08 s^-1 M^-1


## k2 Bayesian Statistics

This section calculates total time spent in encounter before first bound.

In [15]:
encout_time = 0

for sim in stat_bysim.columns:
    mask = (bindsurf_bysim[sim] > fb_dist) & (bindsurf_bysim[sim] < ec_dist)
    #sum_indices = np.sum(bindsurf_bysim.index[mask])
    encout_time += sum(mask)
time_ns_encout = encout_time * 0.01
time_ns_encout

29294.84

In [16]:
# Create a list to store encounter→bound times
encounter_to_bound_times = []

for sim in stat_bysim.columns:
    # Identify encounter and bound frames
    encounter_indices = bindsurf_bysim.index[
        (bindsurf_bysim[sim] > fb_dist) &
        (bindsurf_bysim[sim] <= ec_dist)
    ].tolist()

    bound_indices = bindsurf_bysim.index[
        bindsurf_bysim[sim] < fb_dist
    ].tolist()

    # Handle missing data
    if len(bound_indices) < min_frames or len(encounter_indices) == 0:
        encounter_to_bound_times.append(float('nan'))
        continue

    # --- find first frame that is "fully bound" for min_frames consecutive frames ---
    first_bound = 0
    snapshot = 0
    while first_bound == 0:
        min_met = False
        num_consec = 1
        consec = True

        while (num_consec < min_frames) & consec:
            if bound_indices[snapshot + 1] > bound_indices[snapshot] + 1:
                consec = False
            else:
                num_consec += 1
                snapshot += 1

        if num_consec >= min_frames:
            min_met = True

        if min_met:
            first_bound = bound_indices[snapshot]
            print(first_bound)
        elif snapshot > len(bound_indices) - min_frames - 1:
            first_bound = -1

        snapshot += 1

    # --- compute encounter → bound time ---
    if first_bound < 0:
        encounter_to_bound_times.append(float('nan'))
    else:
        # find first encounter *before* that binding event
        encounter_before_bound = [idx for idx in encounter_indices if idx < first_bound]
        if len(encounter_before_bound) == 0:
            encounter_to_bound_times.append(float('nan'))
        else:
            first_encounter = encounter_before_bound[0]
            time_enc_to_bound = (first_bound - first_encounter) * 0.01  # convert to ns
            encounter_to_bound_times.append(time_enc_to_bound)

# Add as a new column in the same DataFrame
transitions["encounter_to_bound"] = encounter_to_bound_times

48733
68124
23511
43349
66447
77147
19823
33324
10533


In [17]:
transitions

,fully_bound (ns),encounter (ns),encounter_to_bound
0,487.33,47.92,445.10
1,NaN,710.78,NaN
2,NaN,272.33,NaN
3,NaN,125.16,NaN
4,NaN,93.80,NaN
5,NaN,249.61,NaN
6,NaN,293.42,NaN
7,NaN,165.66,NaN
8,NaN,377.78,NaN
9,NaN,727.13,NaN


In [18]:
#tfi = ArkA12_transitions['fully_bound'].sum() #same as the total time considered bound
ti = time_ns_encout #same as the total time in states not yet considered bound

#relv_traj_t = ti
relv_traj_t = ti - transitions["encounter_to_bound"].sum()

relv_traj_t

26444.11

In [19]:
bayesian_k = (num_bound)/relv_traj_t # in ns^-1
var_bayesian_k = (num_bound)/(relv_traj_t**2) # in ns^-2
std_bayesian_k = np.sqrt(var_bayesian_k) # in ns^-1

In [20]:
bayesian_k2 = (bayesian_k)*(10**9)  # unit conversion to M^-1 * s^-1
std_bayesian_k2 = (std_bayesian_k)*(10**9) # unit conversion to s^-1 * M^-1
se_bayesian_k2 = std_bayesian_k2/np.sqrt(num_sims)
print("bayesian_k2 = " + format(bayesian_k2, '.2E') + ' ± ' + format(se_bayesian_k2, '.2E') +' s^-1 M^-1')

bayesian_k2 = 3.40E+05 ± 1.60E+04 s^-1 M^-1


### This section calculates k2 with only last full encounter time until bound (no unbound)

In [21]:
# I got this from Gemini

total_continuous_enc_frames = 0
UNBOUND_THRESHOLD = 100

for sim in stat_bysim.columns:
    # 1. Identify bound indices
    bound_indices = bindsurf_bysim.index[bindsurf_bysim[sim] < fb_dist].tolist()

    # Skip if never bound
    if len(bound_indices) < min_frames:
        continue

    # 2. Find the start of the first stable binding event
    first_bound = 0
    snapshot = 0
    while first_bound == 0 and snapshot < len(bound_indices):
        num_consec = 1
        curr = snapshot
        while (num_consec < min_frames) and (curr + 1 < len(bound_indices)):
            if bound_indices[curr + 1] == bound_indices[curr] + 1:
                num_consec += 1
                curr += 1
            else:
                break
        
        if num_consec >= min_frames:
            first_bound = bound_indices[snapshot]
        else:
            snapshot += 1

    # 3. Trace backward to find the length of the lead-in encounter
    if first_bound > 0:
        # Look at distances before binding in reverse order
        pre_bound_distances = bindsurf_bysim[sim].loc[:first_bound-1].iloc[::-1]
        
        sim_enc_frames = 0
        unbound_counter = 0
        temp_frames = 0 # Track frames since last encounter frame

        for dist in pre_bound_distances:
            if fb_dist < dist <= ec_dist:
                # Add the encounter frame plus any 'flicker' frames we skipped
                sim_enc_frames += 1 + temp_frames
                temp_frames = 0
                unbound_counter = 0
            elif dist > ec_dist:
                unbound_counter += 1
                temp_frames += 1
                if unbound_counter > UNBOUND_THRESHOLD:
                    break
            else:
                # Brief dip into bound zone; treat as encounter progress
                sim_enc_frames += 1 + temp_frames
                temp_frames = 0
                unbound_counter = 0
        
        total_continuous_enc_frames += sim_enc_frames

# Convert total frames to ns
time_ns_encout = total_continuous_enc_frames * 0.01

In [22]:
tfi = transitions_data['fully_bound (ns)'].sum() #same as the total time before the complex is considered bound in the simulations that bind
ti = (num_sims - num_bound) * sim_length_ns #same as the total time of simulations that never go to bound

relv_traj_t = tfi + ti

In [23]:
bayesian_k = (num_bound)/relv_traj_t # in ns^-1
var_bayesian_k = (num_bound)/(relv_traj_t**2) # in ns^-2
std_bayesian_k = np.sqrt(var_bayesian_k) # in ns^-1

In [24]:
bayesian_k2 = (bayesian_k/SH3_conc)*(10**9)  # unit conversion to M^-1 * s^-1
std_bayesian_k2 = (std_bayesian_k/SH3_conc)*(10**9) # unit conversion to s^-1 * M^-1
se_bayesian_k2 = std_bayesian_k2/np.sqrt(50)
print("bayesian_k2 = " + format(bayesian_k2, '.2E') + ' ± ' + format(se_bayesian_k2, '.2E') +' s^-1 M^-1')

bayesian_k2 = 5.27E+07 ± 2.49E+06 s^-1 M^-1


## k_{-1} Bayesian Statistics

In [25]:
bindsurf_bysim

,sim0,sim1,sim2,sim3,sim4,sim5,sim6,sim7,sim8,sim9,...,sim40,sim41,sim42,sim43,sim44,sim45,sim46,sim47,sim48,sim49
0,40.080986,40.437243,38.782986,39.117243,39.712814,40.340557,39.497914,39.815743,40.440243,40.594071,...,35.433714,35.776900,36.277186,35.184700,37.007071,35.964571,35.745757,36.070914,35.574414,35.707471
1,40.189829,41.115929,39.521143,39.440043,39.425971,40.425214,40.804114,40.343371,40.060429,41.488586,...,35.363529,36.875686,36.802314,35.813643,36.313671,35.443943,35.841229,35.780900,35.988171,34.531729
2,39.570643,40.671171,39.630471,38.924800,39.799286,39.876657,40.103557,40.428914,39.574586,41.955714,...,35.158514,36.708300,37.722143,35.690929,36.425400,35.383900,35.458600,36.412129,36.471629,35.023771
3,40.061886,41.004829,38.963729,38.885414,39.037314,40.313871,39.499629,40.764457,39.898900,41.566557,...,34.851843,36.703971,37.470229,34.717214,36.037814,35.327471,36.014386,36.284014,36.033257,34.235843
4,39.164286,40.945300,39.962286,38.393114,40.092829,40.743800,39.826629,40.817843,39.903729,42.575771,...,35.058671,35.384829,37.149429,35.233586,36.050957,35.056529,36.361500,35.717229,36.696500,34.085471
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,9.366029,19.796943,17.835171,18.534700,17.786329,17.447729,20.676386,11.592400,15.604129,20.369300,...,24.998529,19.196514,10.304571,19.141571,21.033957,17.710271,10.638986,18.785357,17.279829,21.091629
99996,9.508471,19.449971,17.928729,18.208329,18.028757,17.178129,20.653900,11.548657,15.515157,20.012729,...,25.204143,19.100243,10.344357,18.828371,20.494243,18.425771,10.790771,18.563800,17.642986,21.159486
99997,9.527071,19.371600,18.105043,19.050314,17.868086,17.445714,21.248829,11.912314,15.614029,20.300829,...,25.380800,19.018743,10.521943,19.114671,21.053529,18.119414,10.706357,18.702014,17.742029,21.459571
99998,9.499129,19.719186,18.226571,19.179971,17.990214,17.592629,21.275700,11.823957,15.222086,20.310486,...,25.420657,19.041229,10.551629,18.176200,20.854514,17.820271,10.762700,18.485343,17.771829,21.393357


In [26]:
# Create a list to store encounter→unbound times
encounter_to_unbound_times = []
encounter_frame_sum = pd.DataFrame(columns=['framesum'])

for sim in stat_bysim.columns:

    # 1. Create a mask for 'encounter'
    is_encounter = (bindsurf_bysim[sim] > fb_dist) & (bindsurf_bysim[sim] <= ec_dist)

    encounter_frame_sum.loc[str(sim), 'framesum'] = is_encounter.sum()
    
    # 2. Identify consecutive blocks of the same state
    # This increments the ID every time the state changes
    block_id = (is_encounter != is_encounter.shift()).cumsum()
    
    # 3. Filter for blocks that are 'encounter' AND have length >= 100
    # We use transform('size') to get the length of each block
    durations = bindsurf_bysim[sim].groupby(block_id).transform('size')
    valid_starts = bindsurf_bysim[sim][(is_encounter) & (durations >= min_frames)]
    
    if not valid_starts.empty:
        # 4. Get the index of the first frame that meets the criteria
        first_idx = valid_starts.index[0]
        
        # 5. Slice the original dataframe from that point onwards
        df_subset = bindsurf_bysim[sim].loc[first_idx:]
    else:
        print("No encounter phase lasted 100 frames or more.")
        
    # Identify encounter and bound frames
    encounter_indices = df_subset.index[
        (df_subset > fb_dist) &
        (df_subset <= ec_dist)
    ].tolist()

    unbound_indices = df_subset.index[
        df_subset > ec_dist
    ].tolist()

    # Handle missing data
    if len(unbound_indices) < min_frames or len(encounter_indices) == 0:
        encounter_to_unbound_times.append(float('nan'))
        continue

    # --- find first frame that is "unbound" for min_frames consecutive frames ---
    first_unbound = 0
    snapshot = 0
    while first_unbound == 0:
        min_met = False
        num_consec = 1
        consec = True

        while (num_consec < min_frames) & consec:
            if unbound_indices[snapshot + 1] > unbound_indices[snapshot] + 1:
                consec = False
            else:
                num_consec += 1
                snapshot += 1

        if num_consec >= min_frames:
            min_met = True

        if min_met:
            first_unbound = unbound_indices[snapshot]
            #print('min_met')
            #print(first_unbound)
            #print(snapshot)
        elif snapshot > len(unbound_indices) - min_frames - 1:
            print(snapshot)
            first_unbound = -1

        snapshot += 1

    # --- compute encounter → unbound time ---
    if first_unbound < 0:
        encounter_to_unbound_times.append(float('nan'))
    else:
        # find first encounter *before* that binding event
        encounter_before_unbound = [idx for idx in encounter_indices if idx < first_unbound]
        if len(encounter_before_unbound) == 0:
            encounter_to_unbound_times.append(float('nan'))
        else:
            #print("test")
            first_encounter = encounter_before_unbound[0]
            time_enc_to_unbound = (first_unbound - first_encounter) * 0.01  # convert to ns
            encounter_to_unbound_times.append(time_enc_to_unbound)

# Add as a new column in the same DataFrame
transitions["encounter_to_unbound"] = encounter_to_unbound_times

34
231
4
565


In [27]:
transitions

,fully_bound (ns),encounter (ns),encounter_to_bound,encounter_to_unbound
0,487.33,47.92,445.10,NaN
1,NaN,710.78,NaN,91.64
2,NaN,272.33,NaN,500.92
3,NaN,125.16,NaN,136.14
4,NaN,93.80,NaN,198.40
5,NaN,249.61,NaN,133.69
6,NaN,293.42,NaN,395.21
7,NaN,165.66,NaN,2.02
8,NaN,377.78,NaN,13.33
9,NaN,727.13,NaN,24.18


In [28]:
sim_length_ns = sim_length * 0.01

In [29]:
encounter_frame_sum

,framesum
sim0,73409
sim1,19251
sim2,64972
sim3,76774
sim4,89701
sim5,67734
sim6,60045
sim7,74552
sim8,58504
sim9,5346


In [30]:
encounter_total_frames = (encounter_frame_sum.reset_index()['framesum'][transitions['encounter_to_unbound'].isnull()]).sum()

In [31]:
tfi = transitions["encounter_to_unbound"].sum() #same as the total time before the complex is considered unbound in the simulations that bind
#ti = (num_sims - num_bound) * sim_length_ns #same as the total time of simulations that never go to unbound
ti = encounter_total_frames*0.01

relv_traj_t = tfi + ti

num_unbound = transitions["encounter_to_unbound"].count()

bayesian_k = (num_unbound)/relv_traj_t # in ns^-1
var_bayesian_k = (num_unbound)/(relv_traj_t**2) # in ns^-2
std_bayesian_k = np.sqrt(var_bayesian_k) # in ns^-1

bayesian_k_1 = (bayesian_k)*(10**9)  # unit conversion to s^-1
std_bayesian_k_1 = (std_bayesian_k)*(10**9) # unit conversion to s^-1
se_bayesian_k_1 = std_bayesian_k_1/np.sqrt(num_sims)
print("bayesian_k_1 = " + format(bayesian_k_1, '.2E') + ' ± ' + format(std_bayesian_k_1, '.2E') +' s^-1')

bayesian_k_1 = 5.10E+06 ± 7.77E+05 s^-1


In [32]:
num_unbound

43

In [33]:
relv_traj_t

8434.39

In [34]:
# encout_time = 0

# for sim in stat_bysim.columns:
#     mask = (bindsurf_bysim[sim] > fb_dist) & (bindsurf_bysim[sim] < ec_dist)
#     #sum_indices = np.sum(arka12_bindsurf_bysim.index[mask])
#     encout_time += sum(mask)
# time_ns_encout = encout_time * 0.01
# #tfi = ArkA12_transitions['fully_bound'].sum() #same as the total time considered bound
# ti = time_ns_encout #same as the total time in states not yet considered bound

# relv_traj_t = ti

# bayesian_k = (num_bound)/relv_traj_t # in ns^-1
# var_bayesian_k = (num_bound)/(relv_traj_t**2) # in ns^-2
# std_bayesian_k = np.sqrt(var_bayesian_k) # in ns^-1
# bayesian_k2 = (bayesian_k/SH3_conc)*(10**9)  # unit conversion to M^-1 * s^-1
# std_bayesian_k2 = (std_bayesian_k/SH3_conc)*(10**9) # unit conversion to s^-1 * M^-1
# se_bayesian_k2 = std_bayesian_k2/np.sqrt(50)

In [35]:
# bayesian_k2

In [36]:
kD_calc = bayesian_k_1/bayesian_k1
kD_calc_err = kD_calc*np.sqrt((std_bayesian_k_1/bayesian_k_1)**2 + (k1_bayesian_err/bayesian_k1)**2)
print("Calculated kD from Bayesian values = " + format(kD_calc, '.2E') + ' ± ' + format(kD_calc_err, '.2E') +' s^-1')

Calculated kD from Bayesian values = 3.62E-03 ± 7.53E-04 s^-1


In [37]:
bayesian_k_1

5098175.44600143

In [38]:
bayesian_k1

1408154555.4391296

In [39]:
std_bayesian_k_1

777464.4668200072

In [40]:
k1_bayesian_err

199143127.02194735

In [67]:
transitions

,fully_bound (ns),encounter (ns),encounter_to_unbound,encounter_to_bound
0,NaN,8.47,NaN,NaN
1,NaN,48.01,7.08,NaN
2,NaN,27.73,NaN,NaN
3,NaN,60.90,14.71,NaN
4,NaN,14.09,12.09,NaN
5,NaN,24.40,NaN,NaN
6,NaN,97.77,21.24,NaN
7,192.22,44.16,4.59,149.97
8,NaN,24.47,NaN,NaN
9,NaN,10.60,973.26,NaN


## Odds Ratio Calculation

This is taken from equation (39) in Ensign and Pande (2009), to determine certainty of processes ocurring at the same rate for two sets of simulations with the same length of simulations time. (refer to paper to make further conclusions based on the odds ratio)

In [4]:
n1 = 9 # number of no salt simulations that reached a state
n2 = 9 # number of salt simulations that reached the same state

R = (2**(n1 + n2 + 1)/np.pi) * ((n1 + n2)/(n1**2 + n2**2)) * (math.factorial(n1)*math.factorial(n2))/math.factorial(n1 + n2) # odds ratio

print(R)

0.3813840980107117


In [42]:
def odds_ratio(k_val): # k_val is a string, thetas here are originally in ns
    unit_conv = 1e6
    if k_val == 'k1':
        n1 = 50
        n2 = 50
        theta1 = 2588.97/unit_conv
        theta2 = 9344.07/unit_conv
    if k_val == 'kon':
        n1 = 9
        n2 = 9
        theta1 = 44753.31/unit_conv
        theta2 = 44909.91/unit_conv
    if k_val == 'k_1':
        n1 = 37
        n2 = 43
        theta1 = 14542.11/unit_conv
        theta2 = 8434.39/unit_conv
        
    n = n1 + n2
    theta = theta1 + theta2

    print(n, theta)
    first_term = (n/theta)/((n1**2/theta1**2) + (n2**2/theta2**2))
    second_term = ((math.factorial(n1)*math.factorial(n2))/math.factorial(n))
    third_term = (theta**(n+1)/(theta1**(n1+1)*theta2**(n2+1)))
    
    R = (2/np.pi) * first_term * second_term * third_term

    return first_term, second_term, third_term, R

In [43]:
odds_ratio('k1')

100 0.011933039999999999


(2.086607064243482e-05,
 9.911653021418339e-30,
 1.5312375526288557e+41,
 20160878.75863664)